# 面试题：混合检索、重排与离线评估怎样连成一条可解释流水线？

## 可以直接复述的回答

混合检索通常让关键词通道捕获精确词，让向量通道召回同义表达，再合并候选而不是直接相加不可比的原始分数。RRF 用名次融合可以减少 BM25 与 cosine 量纲差异，之后的轻量重排器再综合词面、语义、融合排名和新鲜度特征。过滤失效文档与权限门禁必须早于候选装配，否则旧文档即使最后被隐藏，也可能占用召回预算。离线评估至少要同时看 Recall@k、MRR 和 nDCG@k，因为“相关文档进了候选集”和“它排在第一名”是两件事。逐查询结果与候选特征账本比单个平均数更能暴露问题。本题用 10 篇帮助文档与 6 个真实意图，NumPy 手算 cosine、标准库手算 lexical score、RRF、线性重排和三个指标。

## 真实案例

每篇文档包含 tokens、8 维可解释语义向量、freshness 和 active 字段；每条查询有人工相关文档。向量是为教学手工标注的概念坐标，不是线上 embedding，指标也仅描述这组受控样本。

In [1]:
import math  # 导入对数函数以计算词项权重和 nDCG
from collections import Counter  # 导入计数器以统计文档频率
from pprint import pprint  # 导入结构化打印函数以展示候选特征
import numpy as np  # 导入 NumPy 以手算向量余弦相似度
documents = [{"id": "D1", "标题": "退款到账时限", "tokens": ["退款", "到账", "时限", "工作日"], "vector": [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0], "freshness": 1.0, "active": True}, {"id": "D2", "标题": "旧版退钱说明", "tokens": ["退钱", "多久", "退款", "七天"], "vector": [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7, 0.0], "freshness": 0.1, "active": False}, {"id": "D3", "标题": "退款申请路径", "tokens": ["退款", "申请", "售后", "订单"], "vector": [1.0, 0.0, 0.0, 0.2, 0.0, 0.0, 0.1, 0.0], "freshness": 0.9, "active": True}, {"id": "D4", "标题": "电子发票下载", "tokens": ["电子", "发票", "下载", "邮箱"], "vector": [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2], "freshness": 0.8, "active": True}, {"id": "D5", "标题": "账号锁定与登录", "tokens": ["账号", "登录", "锁定", "密码"], "vector": [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.1], "freshness": 0.9, "active": True}, {"id": "D6", "标题": "取消未发货订单", "tokens": ["取消", "订单", "未发货", "关闭"], "vector": [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], "freshness": 1.0, "active": True}, {"id": "D7", "标题": "修改收货地址", "tokens": ["收货", "地址", "修改", "发货"], "vector": [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.7], "freshness": 1.0, "active": True}, {"id": "D8", "标题": "退款凭证要求", "tokens": ["退款", "凭证", "照片", "破损"], "vector": [0.8, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], "freshness": 0.9, "active": True}, {"id": "D9", "标题": "发票抬头与税号", "tokens": ["发票", "抬头", "税号", "公司"], "vector": [0.0, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.8], "freshness": 1.0, "active": True}, {"id": "D10", "标题": "手机号换绑", "tokens": ["账号", "手机", "换绑", "修改"], "vector": [0.0, 0.0, 0.8, 0.0, 0.0, 0.0, 0.0, 0.8], "freshness": 0.8, "active": True}]  # 构造十篇带词面、语义和版本字段的帮助文档
queries = [{"id": "Q1", "问题": "退钱多久", "tokens": ["退钱", "多久"], "vector": [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], "gold": "D1"}, {"id": "Q2", "问题": "抬头税号怎么改", "tokens": ["抬头", "税号"], "vector": [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], "gold": "D9"}, {"id": "Q3", "问题": "账号登录被锁", "tokens": ["账号", "登录", "锁定"], "vector": [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], "gold": "D5"}, {"id": "Q4", "问题": "未发货想撤单", "tokens": ["撤单"], "vector": [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], "gold": "D6"}, {"id": "Q5", "问题": "收货地修改", "tokens": ["收货地", "修改"], "vector": [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.7], "gold": "D7"}, {"id": "Q6", "问题": "退款需要什么证明", "tokens": ["退款", "凭证"], "vector": [0.7, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], "gold": "D8"}]  # 构造六条含同义表达和人工相关文档的查询
print("混合检索文档输入预览：")  # 输出真实案例标题
pprint([{"id": doc["id"], "标题": doc["标题"], "tokens": doc["tokens"], "active": doc["active"], "freshness": doc["freshness"]} for doc in documents])  # 展示十篇文档的检索字段
print("查询与人工相关性：")  # 输出评估集标题
pprint([{key: query[key] for key in ["id", "问题", "tokens", "gold"]} for query in queries])  # 展示六条查询及唯一相关文档

混合检索文档输入预览：
[{'active': True,
  'freshness': 1.0,
  'id': 'D1',
  'tokens': ['退款', '到账', '时限', '工作日'],
  '标题': '退款到账时限'},
 {'active': False,
  'freshness': 0.1,
  'id': 'D2',
  'tokens': ['退钱', '多久', '退款', '七天'],
  '标题': '旧版退钱说明'},
 {'active': True,
  'freshness': 0.9,
  'id': 'D3',
  'tokens': ['退款', '申请', '售后', '订单'],
  '标题': '退款申请路径'},
 {'active': True,
  'freshness': 0.8,
  'id': 'D4',
  'tokens': ['电子', '发票', '下载', '邮箱'],
  '标题': '电子发票下载'},
 {'active': True,
  'freshness': 0.9,
  'id': 'D5',
  'tokens': ['账号', '登录', '锁定', '密码'],
  '标题': '账号锁定与登录'},
 {'active': True,
  'freshness': 1.0,
  'id': 'D6',
  'tokens': ['取消', '订单', '未发货', '关闭'],
  '标题': '取消未发货订单'},
 {'active': True,
  'freshness': 1.0,
  'id': 'D7',
  'tokens': ['收货', '地址', '修改', '发货'],
  '标题': '修改收货地址'},
 {'active': True,
  'freshness': 0.9,
  'id': 'D8',
  'tokens': ['退款', '凭证', '照片', '破损'],
  '标题': '退款凭证要求'},
 {'active': True,
  'freshness': 1.0,
  'id': 'D9',
  'tokens': ['发票', '抬头', '税号', '公司'],
  '标题': '发票抬头与税号'},
 

## Baseline / 基线：只用词面检索且不过滤旧版本

词面通道使用平滑 IDF 累加命中词。它擅长“抬头、税号”这类精确查询，却不理解“撤单≈取消订单”，也会把包含“退钱多久”的旧版 D2 排到第一。

In [2]:
document_frequency = Counter()  # 创建关键词文档频率计数器
for document in documents:  # 遍历全部十篇文档
    for token in set(document["tokens"]):  # 同一文档内的重复词只计一次
        document_frequency[token] += 1  # 累加当前词的文档频率
def lexical_score(query, document):  # 定义可解释的稀有词加权匹配分
    score = 0.0  # 初始化关键词总分
    for token in dict.fromkeys(query["tokens"]):  # 遍历去重后的查询词
        if token in document["tokens"]:  # 只对实际词面命中加分
            score += math.log(1 + (len(documents) + 1) / (document_frequency[token] + 1))  # 用平滑 IDF 奖励稀有词
    return score  # 返回关键词通道分数
def lexical_ranking(query, allowed_only=False):  # 定义词面排序函数
    candidates = [document for document in documents if document["active"] or not allowed_only]  # 根据开关执行版本门禁
    rows = [{"doc_id": document["id"], "score": lexical_score(query, document)} for document in candidates]  # 计算候选词面分数
    return sorted(rows, key=lambda row: (-row["score"], row["doc_id"]))  # 返回稳定的完整排名
baseline_rankings = {query["id"]: lexical_ranking(query, allowed_only=False) for query in queries}  # 运行不过滤旧文档的词面基线
print("Lexical Baseline 的逐查询 Top3：")  # 输出基线排名标题
pprint([{"问题": query["问题"], "gold": query["gold"], "Top3": [(row["doc_id"], round(row["score"], 3)) for row in baseline_rankings[query["id"]][:3]]} for query in queries])  # 展示每条查询的前三名

Lexical Baseline 的逐查询 Top3：
[{'Top3': [('D2', 3.744), ('D1', 0.0), ('D10', 0.0)],
  'gold': 'D1',
  '问题': '退钱多久'},
 {'Top3': [('D9', 3.744), ('D1', 0.0), ('D10', 0.0)],
  'gold': 'D9',
  '问题': '抬头税号怎么改'},
 {'Top3': [('D5', 5.284), ('D10', 1.54), ('D1', 0.0)],
  'gold': 'D5',
  '问题': '账号登录被锁'},
 {'Top3': [('D1', 0.0), ('D10', 0.0), ('D2', 0.0)],
  'gold': 'D6',
  '问题': '未发货想撤单'},
 {'Top3': [('D10', 1.54), ('D7', 1.54), ('D1', 0.0)],
  'gold': 'D7',
  '问题': '收货地修改'},
 {'Top3': [('D8', 3.035), ('D1', 1.163), ('D2', 1.163)],
  'gold': 'D8',
  '问题': '退款需要什么证明'}]


## 手写向量召回、RRF 与线性重排

向量通道显式计算 dot/(norm×norm)。两个通道各取 Top4，RRF 合并后再用 cosine、归一化词面分、RRF 和 freshness 重排；所有计算都只在 active 文档上进行。

In [3]:
def cosine_similarity(left, right):  # 定义 NumPy 余弦相似度
    left_vector = np.asarray(left, dtype=np.float64)  # 把查询概念坐标转换为浮点数组
    right_vector = np.asarray(right, dtype=np.float64)  # 把文档概念坐标转换为浮点数组
    denominator = np.linalg.norm(left_vector) * np.linalg.norm(right_vector)  # 计算两个向量范数的乘积
    return float(left_vector @ right_vector / denominator) if denominator else 0.0  # 计算点积归一化结果并保护零向量
def dense_ranking(query, allowed_only=True):  # 定义语义向量排序函数
    candidates = [document for document in documents if document["active"] or not allowed_only]  # 根据门禁开关选择候选
    rows = [{"doc_id": document["id"], "score": cosine_similarity(query["vector"], document["vector"])} for document in candidates]  # 实算每个候选的余弦分数
    return sorted(rows, key=lambda row: (-row["score"], row["doc_id"]))  # 按相似度和文档编号稳定排序
def reciprocal_rank_fusion(rankings, constant=20, top_k=4):  # 定义两个召回通道的 RRF 候选融合
    fused = {}  # 创建按文档聚合的融合账本
    for channel, ranking in rankings.items():  # 遍历 lexical 与 dense 两个通道
        for rank, row in enumerate(ranking[:top_k], start=1):  # 只融合每个通道前四名
            entry = fused.setdefault(row["doc_id"], {"doc_id": row["doc_id"], "rrf": 0.0, "ranks": {}})  # 初始化候选融合记录
            entry["rrf"] += 1 / (constant + rank)  # 累加基于名次的可比分数
            entry["ranks"][channel] = rank  # 保存候选在当前通道的原始名次
    return sorted(fused.values(), key=lambda row: (-row["rrf"], row["doc_id"]))  # 返回 RRF 排名
print("Q1 的真实向量余弦 Top4：")  # 输出语义召回中间量标题
pprint(dense_ranking(queries[0])[:4])  # 展示同义表达如何召回当前退款文档

Q1 的真实向量余弦 Top4：
[{'doc_id': 'D1', 'score': 0.9986178293325095},
 {'doc_id': 'D3', 'score': 0.7590721152765896},
 {'doc_id': 'D8', 'score': 0.4417261042993862},
 {'doc_id': 'D10', 'score': 0.0}]


In [4]:
document_by_id = {document["id"]: document for document in documents}  # 建立文档编号到字段记录的索引
def hybrid_rerank(query):  # 定义门禁、双路召回、RRF 与线性重排的完整流水线
    lexical = lexical_ranking(query, allowed_only=True)  # 在 active 文档上执行词面召回
    dense = dense_ranking(query, allowed_only=True)  # 在 active 文档上执行语义召回
    fused = reciprocal_rank_fusion({"lexical": lexical, "dense": dense})  # 融合两个通道的前四名
    lexical_scores = {row["doc_id"]: row["score"] for row in lexical}  # 建立候选到词面分数的索引
    dense_scores = {row["doc_id"]: row["score"] for row in dense}  # 建立候选到余弦分数的索引
    maximum_lexical = max(lexical_scores.values()) if lexical_scores else 1.0  # 读取当前查询的最大词面分用于归一化
    maximum_rrf = max(row["rrf"] for row in fused) if fused else 1.0  # 读取最大 RRF 分用于归一化
    reranked = []  # 收集候选的完整重排特征
    for row in fused:  # 遍历 RRF 候选池
        document = document_by_id[row["doc_id"]]  # 读取当前候选的新鲜度等字段
        lexical_normalized = lexical_scores[row["doc_id"]] / maximum_lexical if maximum_lexical > 0 else 0.0  # 安全归一化词面分数
        rrf_normalized = row["rrf"] / maximum_rrf  # 归一化融合名次分数
        dense_score = dense_scores[row["doc_id"]]  # 读取实际余弦相似度
        rerank_score = 0.30 * lexical_normalized + 0.45 * dense_score + 0.15 * rrf_normalized + 0.10 * document["freshness"]  # 用透明权重组合四个重排特征
        reranked.append({"doc_id": row["doc_id"], "lexical": round(lexical_scores[row["doc_id"]], 4), "dense": round(dense_score, 4), "ranks": row["ranks"], "rrf": round(row["rrf"], 5), "freshness": document["freshness"], "rerank": round(rerank_score, 5)})  # 保存可审计候选账本
    return sorted(reranked, key=lambda row: (-row["rerank"], row["doc_id"]))  # 返回最终重排结果
hybrid_rankings = {query["id"]: hybrid_rerank(query) for query in queries}  # 对六条查询运行完整混合流水线
print("Q1 候选融合与重排特征账本：")  # 输出最关键的中间结果标题
pprint(hybrid_rankings["Q1"])  # 展示每个候选的双路名次、分数和最终重排分

Q1 候选融合与重排特征账本：
[{'dense': 0.9986,
  'doc_id': 'D1',
  'freshness': 1.0,
  'lexical': 0.0,
  'ranks': {'dense': 1, 'lexical': 1},
  'rerank': 0.69938,
  'rrf': 0.09524},
 {'dense': 0.7591,
  'doc_id': 'D3',
  'freshness': 0.9,
  'lexical': 0.0,
  'ranks': {'dense': 2, 'lexical': 3},
  'rerank': 0.57165,
  'rrf': 0.08893},
 {'dense': 0.4417,
  'doc_id': 'D8',
  'freshness': 0.9,
  'lexical': 0.0,
  'ranks': {'dense': 3},
  'rerank': 0.35726,
  'rrf': 0.04348},
 {'dense': 0.0,
  'doc_id': 'D10',
  'freshness': 0.8,
  'lexical': 0.0,
  'ranks': {'dense': 4, 'lexical': 2},
  'rerank': 0.21722,
  'rrf': 0.08712},
 {'dense': 0.0,
  'doc_id': 'D4',
  'freshness': 0.8,
  'lexical': 0.0,
  'ranks': {'lexical': 4},
  'rerank': 0.14563,
  'rrf': 0.04167}]


## 离线指标、逐样本结果与结果解读

Recall@3 判断相关文档是否进入前三，MRR 和 nDCG@3 进一步奖励更靠前的位置。只有一个相关文档时，nDCG 仍能直观反映第一名与第三名的差异。下面在同一人工判断集上同时评估词面基线和混合重排。

In [5]:
def evaluate(rankings, cutoff=3):  # 定义 Recall、MRR 与 nDCG 的统一评估函数
    recalls = []  # 收集每条查询的 Recall@k
    reciprocal_ranks = []  # 收集每条查询的倒数排名
    discounted_gains = []  # 收集每条查询的归一化折损增益
    for query in queries:  # 遍历六条人工标注查询
        ranked_ids = [row["doc_id"] for row in rankings[query["id"]]]  # 提取当前方法的文档编号排名
        rank = ranked_ids.index(query["gold"]) + 1 if query["gold"] in ranked_ids else None  # 查找唯一相关文档的实际名次
        recalls.append(1.0 if rank is not None and rank <= cutoff else 0.0)  # 判断相关文档是否进入前 k
        reciprocal_ranks.append(1.0 / rank if rank is not None else 0.0)  # 计算倒数排名
        discounted_gains.append(1.0 / math.log2(rank + 1) if rank is not None and rank <= cutoff else 0.0)  # 计算单相关文档下的 nDCG@k
    return {f"Recall@{cutoff}": sum(recalls) / len(recalls), "MRR": sum(reciprocal_ranks) / len(reciprocal_ranks), f"nDCG@{cutoff}": sum(discounted_gains) / len(discounted_gains)}  # 返回三个查询平均指标
baseline_metrics = evaluate(baseline_rankings)  # 评估不过滤旧版本的词面基线
hybrid_metrics = evaluate(hybrid_rankings)  # 评估门禁后的混合重排方案
comparison_rows = []  # 创建逐查询排名对照表
for query in queries:  # 遍历同一评估集
    baseline_ids = [row["doc_id"] for row in baseline_rankings[query["id"]][:3]]  # 读取基线前三名
    hybrid_ids = [row["doc_id"] for row in hybrid_rankings[query["id"]][:3]]  # 读取混合方案前三名
    comparison_rows.append({"问题": query["问题"], "gold": query["gold"], "Lexical Top3": baseline_ids, "Hybrid Top3": hybrid_ids, "Hybrid Top1正确": hybrid_ids[0] == query["gold"]})  # 保存逐样本候选差异
print("逐查询结果对照：")  # 输出结果表标题
pprint(comparison_rows)  # 展示每条查询的基线和混合前三名
print("离线指标对照：", {"Lexical": baseline_metrics, "Hybrid+Rerank": hybrid_metrics})  # 输出三个同口径指标

逐查询结果对照：
[{'Hybrid Top1正确': True,
  'Hybrid Top3': ['D1', 'D3', 'D8'],
  'Lexical Top3': ['D2', 'D1', 'D10'],
  'gold': 'D1',
  '问题': '退钱多久'},
 {'Hybrid Top1正确': True,
  'Hybrid Top3': ['D9', 'D4', 'D10'],
  'Lexical Top3': ['D9', 'D1', 'D10'],
  'gold': 'D9',
  '问题': '抬头税号怎么改'},
 {'Hybrid Top1正确': True,
  'Hybrid Top3': ['D5', 'D10', 'D1'],
  'Lexical Top3': ['D5', 'D10', 'D1'],
  'gold': 'D5',
  '问题': '账号登录被锁'},
 {'Hybrid Top1正确': True,
  'Hybrid Top3': ['D6', 'D3', 'D1'],
  'Lexical Top3': ['D1', 'D10', 'D2'],
  'gold': 'D6',
  '问题': '未发货想撤单'},
 {'Hybrid Top1正确': True,
  'Hybrid Top3': ['D7', 'D10', 'D9'],
  'Lexical Top3': ['D10', 'D7', 'D1'],
  'gold': 'D7',
  '问题': '收货地修改'},
 {'Hybrid Top1正确': True,
  'Hybrid Top3': ['D8', 'D3', 'D1'],
  'Lexical Top3': ['D8', 'D1', 'D2'],
  'gold': 'D8',
  '问题': '退款需要什么证明'}]
离线指标对照： {'Lexical': {'Recall@3': 0.8333333333333334, 'MRR': 0.6904761904761904, 'nDCG@3': 0.7103099178571526}, 'Hybrid+Rerank': {'Recall@3': 1.0, 'MRR': 1.0, 'nDCG@3': 1.0}}

## 失败案例：原始分数相加且晚过滤旧文档

BM25 类词面分与 cosine 不在同一量纲，直接相加会让精确词面占据不成比例的权重；若同时不过滤 active，旧版 D2 会因“退钱、多久”完全匹配而登顶。修正流程先做版本门禁，再用 RRF 融合名次并重排。

In [6]:
failure_query = queries[0]  # 选择含口语同义词的退款时限问题
unsafe_rows = []  # 创建原始分直接相加的错误候选表
for document in documents:  # 错误地让失效文档也参与候选排序
    lexical = lexical_score(failure_query, document)  # 计算未经归一化的词面分数
    dense = cosine_similarity(failure_query["vector"], document["vector"])  # 计算零到一范围的余弦分数
    unsafe_rows.append({"doc_id": document["id"], "active": document["active"], "lexical": round(lexical, 4), "dense": round(dense, 4), "raw_sum": round(lexical + dense, 4)})  # 直接相加两个不可比分数
unsafe_rows = sorted(unsafe_rows, key=lambda row: (-row["raw_sum"], row["doc_id"]))  # 对错误总分排序
unsafe_top = unsafe_rows[0]  # 读取错误实现的第一名
fixed_top = hybrid_rankings["Q1"][0]  # 读取门禁加 RRF 重排后的第一名
print("失败案例：原始分相加且旧版本参与")  # 输出失败行为标题
pprint(unsafe_rows[:4])  # 展示旧文档如何依靠词面分压过当前文档
print("修正后的第一名：", fixed_top)  # 展示门禁与融合后的正确结果

失败案例：原始分相加且旧版本参与
[{'active': False,
  'dense': 0.9848,
  'doc_id': 'D2',
  'lexical': 3.7436,
  'raw_sum': 4.7284},
 {'active': True,
  'dense': 0.9986,
  'doc_id': 'D1',
  'lexical': 0.0,
  'raw_sum': 0.9986},
 {'active': True,
  'dense': 0.7591,
  'doc_id': 'D3',
  'lexical': 0.0,
  'raw_sum': 0.7591},
 {'active': True,
  'dense': 0.4417,
  'doc_id': 'D8',
  'lexical': 0.0,
  'raw_sum': 0.4417}]
修正后的第一名： {'doc_id': 'D1', 'lexical': 0.0, 'dense': 0.9986, 'ranks': {'lexical': 1, 'dense': 1}, 'rrf': 0.09524, 'freshness': 1.0, 'rerank': 0.69938}


## 生产差距

线上向量应来自经过版本管理的 embedding 模型，并用 ANN 索引权衡 recall 与延迟；lexical 与 dense 还要分别做字段过滤、ACL 和超时降级。重排权重应由带偏差校正的判断集或学习排序训练得到，离线指标需按查询类型分群，并用线上实验校验点击、解决率、延迟和陈旧文档率。

In [7]:
assert len(documents) == 10  # 验证案例包含十篇带版本字段的业务文档
assert unsafe_top["doc_id"] == "D2" and not unsafe_top["active"]  # 验证错误流水线让失效文档登顶
assert fixed_top["doc_id"] == "D1"  # 验证前置门禁与混合重排恢复当前文档
assert hybrid_metrics["MRR"] > baseline_metrics["MRR"]  # 验证混合方案提升第一相关结果位置
assert hybrid_metrics["nDCG@3"] > baseline_metrics["nDCG@3"]  # 验证前三名排序质量得到提升
assert all(row["Hybrid Top1正确"] for row in comparison_rows)  # 验证六条教学查询的第一名均为人工相关文档
print("最小回归测试通过：双路召回、RRF、重排、评估与版本门禁均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：双路召回、RRF、重排、评估与版本门禁均满足预期
